In [2]:
print("All Okay")

All Okay


In [21]:
# Import necessary libraries
import os 
from dotenv import load_dotenv
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import JinaEmbeddings

In [4]:
load_dotenv()

True

In [6]:
groq_key = os.getenv("GROQ_API_KEY")
jina_key = os.getenv("JINA_API_KEY")

print("Environment Variables Loaded ")

Environment Variables Loaded 


In [9]:
# Load the data 

DATA_FILE_PATH = os.path.join("data","hr_policy.txt")

###DATA Ingestion 


In [10]:
text_loader = TextLoader(DATA_FILE_PATH,encoding = "utf-8")

documents = text_loader.load()

print("DATA LOADED")
print(documents)

DATA LOADED
[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='1. Paid Time Off (PTO) & Leave Policy\nPurpose & Scope\nThis policy establishes the rules governing time off from work, ensuring employees receive adequate rest while maintaining operational continuity. It applies to all full-time and part-time employees.\n\nPTO Accrual & Entitlement\nAccrual Rate: Full-time employees accrue Paid Time Off at a rate of 1.66 days per completed calendar month of service, totaling 20 business days per calendar year.\n\nPro-Rating: Employees starting mid-year accrue PTO on a pro-rata basis starting from their official date of joining.\n\nCarry-Over Limit: A maximum of 5 unused PTO days may be carried over into the following calendar year. Unused days exceeding this cap are forfeited on December 31st without cash payout, unless mandated otherwise by local labor laws.\n\nLeave Types & Definitions\nCasual/Vacation Leave: Intended for planned personal time off. Must be requested at 

In [12]:
print(documents[0].page_content)

1. Paid Time Off (PTO) & Leave Policy
Purpose & Scope
This policy establishes the rules governing time off from work, ensuring employees receive adequate rest while maintaining operational continuity. It applies to all full-time and part-time employees.

PTO Accrual & Entitlement
Accrual Rate: Full-time employees accrue Paid Time Off at a rate of 1.66 days per completed calendar month of service, totaling 20 business days per calendar year.

Pro-Rating: Employees starting mid-year accrue PTO on a pro-rata basis starting from their official date of joining.

Carry-Over Limit: A maximum of 5 unused PTO days may be carried over into the following calendar year. Unused days exceeding this cap are forfeited on December 31st without cash payout, unless mandated otherwise by local labor laws.

Leave Types & Definitions
Casual/Vacation Leave: Intended for planned personal time off. Must be requested at least 10 business days in advance for requests exceeding 3 consecutive days.

Sick Leave: In

In [13]:
print(documents[0].metadata)

{'source': 'data\\hr_policy.txt'}


In [16]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print(chunks)
print(len(chunks))

[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='1. Paid Time Off (PTO) & Leave Policy\nPurpose & Scope\nThis policy establishes the rules governing time off from work, ensuring employees receive adequate rest while maintaining operational continuity. It applies to all full-time and part-time employees.\n\nPTO Accrual & Entitlement\nAccrual Rate: Full-time employees accrue Paid Time Off at a rate of 1.66 days per completed calendar month of service, totaling 20 business days per calendar year.'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='Pro-Rating: Employees starting mid-year accrue PTO on a pro-rata basis starting from their official date of joining.\n\nCarry-Over Limit: A maximum of 5 unused PTO days may be carried over into the following calendar year. Unused days exceeding this cap are forfeited on December 31st without cash payout, unless mandated otherwise by local labor laws.'), Document(metadata={'source': 'data\\hr_policy.txt'}, pag

In [20]:
print(chunks[0])

page_content='1. Paid Time Off (PTO) & Leave Policy
Purpose & Scope
This policy establishes the rules governing time off from work, ensuring employees receive adequate rest while maintaining operational continuity. It applies to all full-time and part-time employees.

PTO Accrual & Entitlement
Accrual Rate: Full-time employees accrue Paid Time Off at a rate of 1.66 days per completed calendar month of service, totaling 20 business days per calendar year.' metadata={'source': 'data\\hr_policy.txt'}


In [22]:
#Embedding

In [24]:
embeddings_model = JinaEmbeddings(model_name = "jina-embeddings-v2-base-en")

print("Embd model ready , name is :",embeddings_model.model_name)

Embd model ready , name is : jina-embeddings-v2-base-en


In [25]:
#Store data in Vector Database 

from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(chunks,embeddings_model)

In [26]:
vector_store.index.ntotal

30

In [28]:
test_query  = "how much notice period ?"


## Similarity search 

top_matches  = vector_store.similarity_search(test_query,k=2)
for i, match in enumerate(top_matches,start=1):
    print(f"Match {i}:")
    print(f"Content: {match.page_content}")
    print()


Match 1:
Content: Director Level & Above: 60 calendar days written notice required.

Pay in Lieu of Notice: The company reserves the right to provide base salary payment in lieu of requiring the employee to work out their notice period.

Asset Return Protocol
On or before the final working day, the departing employee must return:

Laptop, peripherals, mobile devices, and security key fobs.

Corporate credit cards and access badges.

All physical and digital confidential materials.

Match 2:
Content: Step 2: Formal written grievance submitted to HRBP if Step 1 is unresolved.

Step 3: Grievance Hearing convened within 7 business days, presided over by HR and an neutral senior executive. Final decision communicated in writing within 5 days post-hearing.

9. Offboarding & Separation Policy
Notice Period Requirements
Probationary Employees: 14 calendar days written notice required from either party.

Regular Full-Time Staff: 30 calendar days written notice required.



LLM

In [31]:
from langchain_groq import ChatGroq


llm = ChatGroq(
    model = "openai/gpt-oss-120b",
    temperature = 0.2,
)

llm.model_name

'openai/gpt-oss-120b'

In [46]:
# Retrieval

k = 3

print("Vector store is ready for retrieval")

Vector store is ready for retrieval


In [47]:
question = "what is the notice period?"

matching_docs = vector_store.similarity_search(question, k=k)

print("Matching documents found:", len(matching_docs))

Matching documents found: 3


In [48]:
for i, doc in enumerate(matching_docs, start=1):
    print("Document", i)
    print(doc.page_content)
    print("-" * 50)

Document 1
Director Level & Above: 60 calendar days written notice required.

Pay in Lieu of Notice: The company reserves the right to provide base salary payment in lieu of requiring the employee to work out their notice period.

Asset Return Protocol
On or before the final working day, the departing employee must return:

Laptop, peripherals, mobile devices, and security key fobs.

Corporate credit cards and access badges.

All physical and digital confidential materials.
--------------------------------------------------
Document 2
Step 2: Formal written grievance submitted to HRBP if Step 1 is unresolved.

Step 3: Grievance Hearing convened within 7 business days, presided over by HR and an neutral senior executive. Final decision communicated in writing within 5 days post-hearing.

9. Offboarding & Separation Policy
Notice Period Requirements
Probationary Employees: 14 calendar days written notice required from either party.

Regular Full-Time Staff: 30 calendar days written notic

In [49]:
context = "\n\n".join(doc.page_content for doc in matching_docs)

prompt = f"""
You are a helpful HR assistant.
Answer the question using only the context given below.
If the answer is not in the context, say "I don't know".

Context:
{context}

Question:
{question}
"""

response = llm.invoke(prompt)

print(response.content)

The required notice period depends on the employee’s classification:

- **Director level and above:** 60 calendar days written notice  
- **Probationary employees:** 14 calendar days written notice (from either party)  
- **Regular full‑time staff:** 30 calendar days written notice  

If you fall into one of these groups, use the corresponding notice period.


In [50]:
def ask_hr_question(question):
    matching_docs = vector_store.similarity_search(question, k=k)
    context = "\n\n".join(doc.page_content for doc in matching_docs)

    prompt = f"""
You are a helpful HR assistant.
Answer the question using only the context given below.
If the answer is not in the context, say "I don't know".

Context:
{context}

Question:
{question}
"""

    response = llm.invoke(prompt)
    return response.content

In [51]:
answer = ask_hr_question("how many sick leaves are allowed?")

print(answer)

Employees are entitled to **10 paid sick days per calendar year**.
